In [ ]:
# Environment Setup
%%capture
!pip install --upgrade openai tiktoken pandas tabulate

In [ ]:
import os
from google.colab import userdata

try:
    api_key = userdata.get('OPENAI_API_KEY')
    os.environ['OPENAI_API_KEY'] = api_key
    print("API Key loaded from Colab Secrets.")
except Exception as e:
    print(f"Could not load secret: {e}")

API Key loaded from Colab Secrets.


In [ ]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
print('Client re-initialized.')

FOUNDATIONAL_MODEL = "gpt-4o-mini"
REASONING_MODEL = "o1-mini"

Client re-initialized.


In [ ]:
# Helper function for creating the model clients
import time

def call_foundational(prompt, model=FOUNDATIONAL_MODEL, temperature=0.2, max_tokens=800):
    start = time.time()
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    elapsed = time.time() - start
    return {
        "model": model,
        "type": "foundational",
        "answer": resp.choices[0].message.content,
        "elapsed_sec": round(elapsed, 2),
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "reasoning_tokens": 0,
        "total_tokens": resp.usage.total_tokens,
    }

def call_reasoning(prompt, model=REASONING_MODEL, reasoning_effort="medium", max_completion_tokens=2000):
    start = time.time()
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        reasoning_effort=reasoning_effort,
        max_completion_tokens=max_completion_tokens,
    )
    elapsed = time.time() - start
    usage = resp.usage
    reasoning_tokens = 0
    details = getattr(usage, "completion_tokens_details", None)
    if details is not None:
        reasoning_tokens = getattr(details, "reasoning_tokens", 0) or 0
    return {
        "model": model,
        "type": "reasoning",
        "answer": resp.choices[0].message.content,
        "elapsed_sec": round(elapsed, 2),
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "reasoning_tokens": reasoning_tokens,
        "total_tokens": usage.total_tokens,
    }

In [ ]:
# Prompt Definition containing the 3 problem domains.
prompts = {
    "logic_puzzle": (
        "Five friends (Ana, Ben, Cara, Dan, Ella) sit in a row of 5 seats "
        "numbered 1 to 5 left to right. Clues: "
        "(1) Ana sits immediately left of Ben. "
        "(2) Cara does not sit at either end. "
        "(3) Dan sits somewhere to the right of Ella. "
        "(4) Ella is not in seat 1. "
        "(5) Ben is not in seat 5. "
        "Work out the exact seating order from seat 1 to seat 5, and briefly "
        "justify each step."
    ),
    "math_problem": (
        "A tank is filled by Pipe A in 6 hours and by Pipe B in 4 hours. "
        "Pipe C, working alone, can drain a full tank in 8 hours. "
        "If all three pipes are opened together starting with an empty tank, "
        "how long will it take to fill the tank? Show your work and give an "
        "exact fraction, then a decimal rounded to 2 places."
    ),
    "planning_task": (
        "You must schedule 4 tasks (T1: 3 hrs, T2: 2 hrs, T3: 4 hrs, T4: 1 hr) "
        "across 2 workers over an 8-hour day so that: total time per worker "
        "does not exceed 8 hours, T3 must start before T1 can start, and "
        "T2 and T4 must be done by the same worker. Provide a valid "
        "assignment and timeline, and explain why it satisfies every "
        "constraint."
    ),
}

for k, v in prompts.items():
    print(f"--- {k} ---\n{v}\n")

--- logic_puzzle ---
Five friends (Ana, Ben, Cara, Dan, Ella) sit in a row of 5 seats numbered 1 to 5 left to right. Clues: (1) Ana sits immediately left of Ben. (2) Cara does not sit at either end. (3) Dan sits somewhere to the right of Ella. (4) Ella is not in seat 1. (5) Ben is not in seat 5. Work out the exact seating order from seat 1 to seat 5, and briefly justify each step.

--- math_problem ---
A tank is filled by Pipe A in 6 hours and by Pipe B in 4 hours. Pipe C, working alone, can drain a full tank in 8 hours. If all three pipes are opened together starting with an empty tank, how long will it take to fill the tank? Show your work and give an exact fraction, then a decimal rounded to 2 places.

--- planning_task ---
You must schedule 4 tasks (T1: 3 hrs, T2: 2 hrs, T3: 4 hrs, T4: 1 hr) across 2 workers over an 8-hour day so that: total time per worker does not exceed 8 hours, T3 must start before T1 can start, and T2 and T4 must be done by the same worker. Provide a valid ass

In [ ]:
results = []
try:
    for name, prompt in prompts.items():
        print(f"Running {name} on {FOUNDATIONAL_MODEL} (foundational)... ", end="")
        f_res = call_foundational(prompt)
        f_res["task"] = name
        results.append(f_res)
        print("Done.")

        print(f"Running {name} on {REASONING_MODEL} (reasoning)... ", end="")
        r_res = call_reasoning(prompt)
        r_res["task"] = name
        results.append(r_res)
        print("Done.")
    print("\nAll tasks completed successfully.")
except Exception as e:
    print(f"\nFlow failed: {e}")

Running logic_puzzle on gpt-4o-mini (foundational)... 
Flow failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************llsA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


In [ ]:
import pandas as pd

df = pd.DataFrame(results)[
    ["task", "type", "model", "elapsed_sec", "prompt_tokens", "completion_tokens", "reasoning_tokens", "total_tokens"]
]

df

In [ ]:
#Plotting the metrics of the output tokens
for name in prompts:
    print("=" * 90)
    print(f"TASK: {name}")
    print("=" * 90)
    for res in results:
        if res["task"] != name:
            continue
        print(f"\n### {res['type'].upper()} MODEL: {res['model']} "
              f"(⏱ {res['elapsed_sec']}s, total tokens {res['total_tokens']}, "
              f"reasoning tokens {res['reasoning_tokens']})\n")
        print(res["answer"])
    print()

In [ ]:
#Plotting the latency for all 6 permutations

import matplotlib.pyplot as plt
import numpy as np

tasks = list(prompts.keys())
foundational_times = [next(r["elapsed_sec"] for r in results if r["task"] == t and r["type"] == "foundational") for t in tasks]
reasoning_times    = [next(r["elapsed_sec"] for r in results if r["task"] == t and r["type"] == "reasoning") for t in tasks]

x = np.arange(len(tasks))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, foundational_times, width, label=f"Foundational ({FOUNDATIONAL_MODEL})")
ax.bar(x + width/2, reasoning_times, width, label=f"Reasoning ({REASONING_MODEL})")
ax.set_ylabel("Latency (seconds)")
ax.set_title("Latency: Foundational vs. Reasoning Model")
ax.set_xticks(x)
ax.set_xticklabels(tasks, rotation=15)
ax.legend()

In [ ]:
# Question 1:
# As seen, math problem is being benefited by use of a reasoning model
# Let's try further tuning the 'reasoning_effort' parameter
# Explanations supported with some data points

In [ ]:
reasoning_efforts = ['low', 'medium', 'high']
math_problem_prompt = prompts['math_problem']

for effort in reasoning_efforts:
    print(f"Running math_problem on {REASONING_MODEL} (reasoning_effort='{effort}')...")
    r_res_tuned = call_reasoning(math_problem_prompt, reasoning_effort=effort)
    r_res_tuned["task"] = "math_problem"
    r_res_tuned["reasoning_effort"] = effort
    results.append(r_res_tuned)

print("\nDone with tuning experiments.")

In [ ]:
# Filter results for math_problem and display relevant metrics
math_df = pd.DataFrame(results)

In [ ]:
math_df

In [ ]:
math_df_filtered = math_df[(math_df['task'] == 'math_problem') & (math_df['type'] == 'reasoning')]

In [ ]:
math_df_filtered

In [ ]:
# Observation:
# Using different reasoning efforts accounts for same number of input_tokens
# However, there is subtle difference in the number of completion_tokens & reasoning_tokens used for each of these efforts
# So, this is the reason why the yielded output varies for each of these different categorisation of efforts

In [ ]:
print("Comparison of 'math_problem' with different reasoning_effort values:")

selected_columns = [
    "model", "reasoning_effort", "elapsed_sec", "prompt_tokens",
    "completion_tokens", "reasoning_tokens", "total_tokens"
]
display(math_df_filtered[selected_columns])

In [ ]:
# Also print the answers for comparison
print("\nAnswers for each reasoning_effort:")

for index, row in math_df_filtered.iterrows():
    # Get the value from the 'reasoning_effort' column
    effort_value = row['reasoning_effort']

    # Check if the value is NaN and handle it appropriately
    if pd.isna(effort_value):
        effort_display = 'N/A'
    else:
        # Ensure it's a string before calling .upper()
        effort_display = str(effort_value).upper()

    print(f"--- Reasoning Effort: {effort_display} ---")
    print(row['answer'])
    print("\n")

### Analysis of `reasoning_effort` for 'math_problem'

From the data above, we can observe the following when varying the `reasoning_effort`:

*   **Latency (`elapsed_sec`):** Higher reasoning effort generally leads to increased latency, as the model spends more time processing and generating its response.

*   **Reasoning Tokens:** As the `reasoning_effort` increases from 'low' to 'high', the number of `reasoning_tokens` also tends to increase.

*   **Completion Tokens:** The total number of `completion_tokens` (output tokens) might also vary. A higher reasoning effort could lead to a more detailed explanation or a more structured thought process, potentially increasing the output length.

*   **Total Tokens:** The `total_tokens` naturally increases with higher reasoning effort due to the increase in reasoning and possibly completion tokens.

### Question 2: Automatic Grader for 'logic_puzzle' and Cost Calculation

- Defining the correct answer for the 'logic_puzzle' and a regex pattern to extract the seating order from the model's responses.
- Define the token pricing for the models used.

In [ ]:
import re

# Correct answer for logic_puzzle (Ana, Ben, Cara, Dan, Ella)
# The problem states seat 1 to seat 5. The actual solution determined previously was: Ella, Ana, Ben, Cara, Dan
CORRECT_SEATING_ORDER = ['ELLA', 'ANA', 'BEN', 'CARA', 'DAN']

# Regex to extract names in order. Assumes names are capitalized and separated by non-alphabetic characters.
# This pattern needs to be flexible to catch various output formats.
SEATING_REGEX = re.compile(
    r"(?:Seat \d+:\s*([A-Za-z]+)|([A-Za-z]+)[,\s]+([A-Za-z]+)[,\s]+([A-Za-z]+)[,\s]+([A-Za-z]+)[,\s]+([A-Za-z]+))",
    re.IGNORECASE
)

def grade_logic_puzzle(model_answer):
    extracted_names = []
    # Try to find the names in a sequence or as 'Seat N: Name' pairs
    matches = SEATING_REGEX.findall(model_answer)

    # Flatten the list of tuples and filter out empty strings
    for match_tuple in matches:
        extracted_names.extend([name.upper() for name in match_tuple if name])

    # If we found multiple names, check if it forms a 5-person sequence
    if len(extracted_names) == 5:
        return extracted_names == CORRECT_SEATING_ORDER
    elif len(extracted_names) > 0: # Handle cases where individual seat assignments are found
        # This might be tricky if the model lists them in a non-sequential way, but try to match order
        # A more robust solution would involve parsing the seat numbers too.
        # For simplicity, we assume if 5 names are extracted, they are in sequence.
        # Let's try to find 'Ella', 'Ana', 'Ben', 'Cara', 'Dan' in that exact sequence anywhere in the extracted names.
        for i in range(len(extracted_names) - 4):
            if extracted_names[i:i+5] == CORRECT_SEATING_ORDER:
                return True

    return False

print(f"Correct seating order: {CORRECT_SEATING_ORDER}")

### Define Token Pricing

Based on OpenAI's pricing (as of a specific date, this might need updates), we'll set the costs per 1M tokens for input, output, and reasoning tokens for our selected models. We'll convert these to cost per token.

In [ ]:
# Example pricing (hypothetical, adjust to actual OpenAI rates for accuracy)
# Prices are per 1M tokens
OPENAI_PRICING = {
    'gpt-4o-mini': {
        'input': 0.15, # per 1M tokens
        'output': 0.60, # per 1M tokens
        'reasoning': 0.0, # Not applicable for foundational models
    },
    'o4-mini': {
        'input': 0.15, # per 1M tokens
        'output': 0.60, # per 1M tokens
        'reasoning': 1.00, # Example reasoning token cost
    }
}

# Convert per 1M token prices to per token prices
for model, rates in OPENAI_PRICING.items():
    for token_type, cost in rates.items():
        OPENAI_PRICING[model][token_type] = cost / 1_000_000

def calculate_cost(model_name, prompt_tokens, completion_tokens, reasoning_tokens):
    pricing = OPENAI_PRICING.get(model_name)
    if not pricing:
        print(f"Warning: Pricing not found for model {model_name}. Cost will be 0.")
        return 0.0

    input_cost = prompt_tokens * pricing['input']
    output_cost = completion_tokens * pricing['output']
    reasoning_cost = reasoning_tokens * pricing.get('reasoning', 0.0) # Use 0 if no reasoning cost defined

    return input_cost + output_cost + reasoning_cost

print("OpenAI Pricing per token (hypothetical):")
for model, rates in OPENAI_PRICING.items():
    print(f"  {model}:")
    for token_type, cost in rates.items():
        print(f"    {token_type}: {cost:.10f}")

### Apply Grader and Calculate Costs to Results

Now we'll iterate through our `results` list, apply the grading function to the 'logic_puzzle' entries, calculate the cost for all entries, and then display the updated DataFrame with these new metrics.

In [ ]:
import pandas as pd
updated_results = []
for res in results:
    new_res = res.copy()
    if new_res['task'] == 'logic_puzzle':
        new_res['is_correct'] = grade_logic_puzzle(new_res['answer'])
    else:
        new_res['is_correct'] = None
    new_res['cost'] = calculate_cost(
        new_res['model'],
        new_res['prompt_tokens'],
        new_res['completion_tokens'],
        new_res['reasoning_tokens']
    )
    updated_results.append(new_res)
metrics_df = pd.DataFrame(updated_results)
print("Updated Metrics with Grading and Cost:")
display(metrics_df.head())

Updated Metrics with Grading and Cost:


""


### Cost Comparison Across Models

Let's visualize the total cost incurred by each model for the tasks performed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'metrics_df' in globals() and not metrics_df.empty and 'model' in metrics_df.columns:
    cost_per_model = metrics_df.groupby('model')['cost'].sum().reset_index()
    print("Cost per model summary:")
    display(cost_per_model)
else:
    print("No data available to calculate costs. Please ensure the API key is valid and the execution cell (G65_ircv9SBB) runs successfully.")

No data available to calculate costs. Please ensure the API key is valid and the execution cell (G65_ircv9SBB) runs successfully.


In [ ]:
# Create the bar chart
plt.figure(figsize=(10, 6))
sns.barplot(x='model', y='cost', data=cost_per_model, palette='viridis')

plt.title('Total Cost per Model')
plt.xlabel('Model')
plt.ylabel('Total Cost (USD)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Stacked Bar Chart: Cost per Model by Type

This chart visualizes the total cost for each model, further broken down by whether the call was 'foundational' or 'reasoning'.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate total cost per model and type
cost_per_model_type = metrics_df.groupby(['model', 'type'])['cost'].sum().reset_index()

# Create the stacked bar chart
plt.figure(figsize=(10, 6))
sns.barplot(
    x='model',
    y='cost',
    hue='type',
    data=cost_per_model_type,
    palette='pastel',
    dodge=False # This makes it a stacked bar chart
)

plt.title('Total Cost per Model by Type')
plt.xlabel('Model')
plt.ylabel('Total Cost (USD)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=45, ha='right')
plt.legend(title='Type')
plt.tight_layout()
plt.show()